"This notebook performs GWAS-based signal analysis to identify significant genomic regions associated with target traits. It integrates statistical GWAS results with taglotype (haplotype) information to detect shared signals—genetic variants that are both statistically significant and structurally supported.

The workflow includes filtering significant variants, grouping them into genomic regions, evaluating taglotype support, and ranking regions based on signal strength and consistency. The output highlights the most reliable candidate regions for further biological interpretation and downstream analysis".


In [0]:
import pandas as pd
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()


In [0]:
import sys
sys.path.append("/Volumes/bmqg/default_bronze/fatemeh/final_project/modules")
import importlib
import shared_signal

importlib.reload(shared_signal)
from shared_signal import run_pipeline, get_shared_signals

In [0]:
import yaml 
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import ast, re
from scipy.stats import spearmanr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

# paths / data
GWAS_TABLE = CONFIG["data"]["gwas_table"]

TAGLO_TABLE =CONFIG["paths"]["TAGLO_TABLE"]
PATHS = CONFIG["paths"]
pheno = pd.read_csv(PATHS["aroma_matrix"])


result = run_pipeline(
    spark=spark,   # 
    gwas_table=GWAS_TABLE,
    taglo_table=TAGLO_TABLE,
    pheno_df=pheno,
    phenotype_col="aroma_score"
)
display(result.head())
shared_signals = get_shared_signals(
    spark=spark,
    gwas_table=GWAS_TABLE,
    p_thresh=1e-6
)

display(shared_signals)

trait,chrom,window,n_unique_pos,leader_start,leader_p,leader_nlp,taglo_id1,taglo_id2,taglo_id3,taglo_id4,n_taglo,taglo_ids,consecutive_run_len,best_consecutive_run,rank_in_trait
"1-Hexanol, 2-ethyl-",ST4.03ch01,10500000,6,10800000,1.2107140561631726E-22,21.916958415481265,27159,27166,0,0,28,"List(27168, 27169, 27170, 27171, 27109, 27173, 27110, 27111, 27112, 27113, 27054, 27214, 27281, 27313, 27282, 27283, 27221, 27062, 27222, 27159, 27223, 27224, 27225, 27226, 27322, 27165, 27166, 27167)",7,"List(27165, 27166, 27167, 27168, 27169, 27170, 27171)",1
Benzyl alcohol,ST4.03ch01,10500000,6,10800000,1.2107140561631726E-22,21.916958415481265,27159,27166,0,0,28,"List(27168, 27169, 27170, 27171, 27109, 27173, 27110, 27111, 27112, 27113, 27054, 27214, 27281, 27313, 27282, 27283, 27221, 27062, 27222, 27159, 27223, 27224, 27225, 27226, 27322, 27165, 27166, 27167)",7,"List(27165, 27166, 27167, 27168, 27169, 27170, 27171)",1
"1-Hexanol, 2-ethyl-",ST4.03ch12,49000000,7,49300000,2.1655760490205676E-16,15.664426560449465,555481,0,0,0,27,"List(555496, 555497, 555498, 555562, 555596, 555598, 555490, 555491, 555492, 555397, 555493, 555494, 555399, 555495, 555559, 555289, 555449, 555481, 555290, 555322, 555451, 555324, 555505, 555506, 555283, 555315, 555284)",9,"List(555490, 555491, 555492, 555493, 555494, 555495, 555496, 555497, 555498)",2
Benzyl alcohol,ST4.03ch12,49000000,7,49300000,2.1655760490205676E-16,15.664426560449465,555481,0,0,0,27,"List(555496, 555497, 555498, 555562, 555596, 555598, 555490, 555491, 555492, 555397, 555493, 555494, 555399, 555495, 555559, 555289, 555449, 555481, 555290, 555322, 555451, 555324, 555505, 555506, 555283, 555315, 555284)",9,"List(555490, 555491, 555492, 555493, 555494, 555495, 555496, 555497, 555498)",2
"1-Hexanol, 2-ethyl-",ST4.03ch05,42500000,7,42650000,3.718734846967775E-14,13.42960478657892,251069,0,0,0,18,"List(251075, 251155, 251154, 250913, 250912, 251072, 251280, 251239, 251110, 251109, 251076, 251284, 251240, 251071, 251198, 251069, 251085, 251197)",2,"List(250912, 250913)",3


trait,chrom,start,p_wald,taglo_id1,taglo_id2,taglo_id3,taglo_id4,taglo_id,nlp
Propanal,ST4.03ch04,70350000,4.276978E-8,219640,219642,219643,0,219640,7.368862977515886
Propanal,ST4.03ch04,70350000,4.276978E-8,219640,219642,219643,0,219642,7.368862977515886
Propanal,ST4.03ch04,70350000,4.276978E-8,219640,219642,219643,0,219643,7.368862977515886
Propanal,ST4.03ch03,56450000,5.693323E-8,159568,159570,159579,0,159568,7.244634165449816
Propanal,ST4.03ch03,56450000,5.693323E-8,159568,159570,159579,0,159570,7.244634165449816
Propanal,ST4.03ch03,56450000,5.693323E-8,159568,159570,159579,0,159579,7.244634165449816
Propanal,ST4.03ch03,56450000,2.309193E-7,159568,159570,159576,159579,159568,6.636539771614329
Propanal,ST4.03ch03,56450000,2.309193E-7,159568,159570,159576,159579,159570,6.636539771614329
Propanal,ST4.03ch03,56450000,2.309193E-7,159568,159570,159576,159579,159576,6.636539771614329
Propanal,ST4.03ch03,56450000,2.309193E-7,159568,159570,159576,159579,159579,6.636539771614329
